# Reproduce the Germany 2010 dataset from a fresh checkout

This notebook walks through every step needed to regenerate the dataset published with the paper: archetypes, electricity profiles, occupancy, weather, and the heating/cooling simulation. Run cells in order.

## What you need *before* running this notebook

Two inputs are not bundled with this repository for licensing/size reasons. **You must place them on disk yourself**:

| Input | Where it goes | Source |
| --- | --- | --- |
| HTW Berlin 15-min electricity profiles (74 CSVs) | `input/electricity/15min/<annual_kwh>.csv` | HTW Berlin residential load profile dataset, Tjaden et al. — request from HTW Berlin / dataset DOI. |
| German 10 km UTM grid GeoPackage | `input/grid/DE_Grid_ETRS89-UTM32_10km.gpkg` | Federal Agency for Cartography and Geodesy (BKG): https://gdz.bkg.bund.de/index.php/default/geographische-gitter-fur-deutschland-in-utm-projektion-geogitter-utm.html |

The remaining inputs are produced or downloaded **automatically** by the pipeline:

| Input | How it is obtained |
| --- | --- |
| 11 building archetypes (`output/B.parquet`) | Generated by `TEASERArchetypeProvider` from `input/archetypes.csv` (bundled) using TEASER + the TABULA-DE construction database (bundled with TEASER). |
| Per-grid-cell weather (`input/weather/2010/loc####.parquet`) | Downloaded by `OpenMeteoWeatherProvider` from https://archive-api.open-meteo.com — free, no API key, rate-limited (~4 hours total for 4,045 locations). |
| Occupancy (`output/O.parquet`) | Derived from the electricity profiles by `GeoMAOccupancyProvider`. |
| Heating/cooling demand (`output/HC/loc####/hc_arch##.parquet`) | Computed by EnTiSe's R1C1 model in the simulation step. |

## Environment

From the repository root:

```bash
pip install uv
uv sync
uv run jupyter notebook examples/03_reproduce_paper.ipynb
```

## 0. Pre-flight check

In [1]:
import sys, os
from pathlib import Path

# Repo root (parent of this notebook's folder).
ROOT = Path.cwd().resolve()
if ROOT.name == 'examples':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f'Working directory: {ROOT}')

missing = []
if not (ROOT / 'input/electricity/15min').exists() or not list((ROOT/'input/electricity/15min').glob('*.csv')):
    missing.append('input/electricity/15min/*.csv  (HTW Berlin profiles — see README above)')
if not (ROOT / 'input/grid/DE_Grid_ETRS89-UTM32_10km.gpkg').exists():
    missing.append('input/grid/DE_Grid_ETRS89-UTM32_10km.gpkg  (BKG 10 km grid — see README above)')

if missing:
    print('MISSING INPUTS:')
    for m in missing:
        print(f'  - {m}')
    raise SystemExit('Place the missing files and re-run this cell.')
print('All required inputs present.')

Working directory: C:\Users\ge23nur\Projects\uni\coupled_ts_paper
All required inputs present.


## 1. Run the full pipeline

Everything from here is driven by `config/germany_2010.yml`. The orchestrator runs the four provider steps (archetypes / electricity / occupancy / weather) and then the thermal simulation.

On a fresh checkout this cell takes hours: weather download is rate-limited (Open-Meteo allows ~600 calls/min, so ~7 minutes minimum for 4,045 locations), and the simulation runs ~3.6 M building-years. Intermediate outputs are checkpointed per file, so re-running picks up where it left off.

In [2]:
from src.pipeline import run

result = run(
    Path('config/germany_2010.yml'),
    overwrite=False,           # idempotent re-runs skip done steps
    skip_weather_fetch=False,  # turn on if you've already cached weather
    skip_simulation=False,     # turn on if you only want the input parquets
)
print(result)

Fetching weather: 100%|██████████| 4045/4045 [00:07<00:00, 553.53loc/s, skipped=4045, written=0]

KeyboardInterrupt



## 2. Quick checks

Read the four input parquets and a few simulation outputs to confirm shapes and value ranges.

In [3]:
import pandas as pd

B = pd.read_parquet('output/B.parquet')
E = pd.read_parquet('output/E.parquet')
O = pd.read_parquet('output/O.parquet')
loc = pd.read_csv('output/location_mapping.csv')

print(f'B (archetypes):   {len(B):,} rows, columns={list(B.columns)}')
print(f'E (electricity):  {len(E):,} rows, {E.profile_id.nunique()} profiles')
print(f'O (occupancy):    {len(O):,} rows, occupied={O.occupied.mean()*100:.1f}%')
print(f'Locations:        {len(loc):,}')

B.head()

B (archetypes):   11 rows, columns=['archetype_id', 'construction_year', 'area_m2', 'n_floors', 'height_floor_m', 'thermal_resistance', 'thermal_capacitance']
E (electricity):  648,240 rows, 74 profiles
O (occupancy):    648,240 rows, occupied=45.8%
Locations:        4,045


,archetype_id,construction_year,area_m2,n_floors,height_floor_m,thermal_resistance,thermal_capacitance
0,1,1859,219.0,2,2.5,0.000951,4.818744e+07
1,2,1860,142.0,2,2.5,0.001647,7.966792e+07
2,3,1919,303.0,2,2.5,0.001012,9.119222e+07
3,4,1949,111.0,2,2.5,0.002098,3.178749e+07
4,5,1958,121.0,2,2.5,0.001810,4.861146e+07


In [4]:
# Spot-check one weather file and one HC simulation output.
import pandas as pd
from pathlib import Path

w = pd.read_parquet('input/weather/2010/loc0001.parquet')
print(f'Weather loc0001: {len(w):,} rows, T_air range [{w.air_temperature.min():.1f}, {w.air_temperature.max():.1f}] C')

hc_files = sorted(Path('output/HC/loc0001').glob('hc_arch*.parquet'))
if hc_files:
    hc = pd.read_parquet(hc_files[0])
    print(f'HC {hc_files[0].name}: {len(hc):,} rows, heating={hc.q_heat_w.sum()/1000:.0f} kWh, cooling={hc.q_cool_w.sum()/1000:.0f} kWh')
else:
    print('(simulation output not present — re-run section 1 with skip_simulation=False)')

Weather loc0001: 8,760 rows, T_air range [-24.1, 25.1] C
HC hc_arch01.parquet: 648,240 rows, heating=12479448 kWh, cooling=7 kWh


## 3. Reproducing only part of the pipeline

Each provider can be invoked directly. For example, to regenerate only the archetypes table:

In [5]:
from src.providers import TEASERArchetypeProvider

prov = TEASERArchetypeProvider(archetypes_csv=Path('input/archetypes.csv'))
df = prov.get_archetypes()
df

,archetype_id,construction_year,area_m2,n_floors,height_floor_m,thermal_resistance,thermal_capacitance
0,1,1859,219.0,2,2.5,0.000951,4.818744e+07
1,2,1860,142.0,2,2.5,0.001647,7.966792e+07
2,3,1919,303.0,2,2.5,0.001012,9.119222e+07
3,4,1949,111.0,2,2.5,0.002098,3.178749e+07
4,5,1958,121.0,2,2.5,0.001810,4.861146e+07
5,6,1969,173.0,2,2.5,0.001969,1.403882e+08
6,7,1979,216.0,2,2.5,0.002876,4.845635e+07
7,8,1984,150.0,2,2.5,0.003552,2.010494e+07
8,9,1995,122.0,2,2.5,0.005663,3.528212e+07
9,10,2002,147.0,2,2.5,0.006758,4.543477e+07
